# Optuna



In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import multiprocessing
from sklearn.metrics import roc_auc_score

In [ ]:
## we start again by reading our data

eye_movements = pd.read_csv("../data/eye_movements_aggregated.csv")
eye_movements.head()

In [ ]:
eye_movements.shape

In [ ]:
eye_features = eye_movements.loc[:,['fixcount', 'firstPassCnt', 'P1stFixation', 'P2stFixation',
       'prevFixDur', 'firstfixDur', 'firstPassFixDur', 'nextFixDur',
       'firstSaccLen', 'lastSaccLen', 'prevFixPos', 'landingPos', 'leavingPos',
       'totalFixDur', 'meanFixDur', 'nRegressFrom', 'regressLen',
       'nextWordRegress', 'regressDur', 'pupilDiamMax', 'pupilDiamLag',
       'timePrtctg']]
labels = eye_movements['target'].astype(int)

In [ ]:
##let's start by splitting the data into a train and a validation set
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(eye_features, labels, test_size=0.2, random_state=123)


we train a model to gain an idea of baseline performance 

In [ ]:
%%time 
import xgboost as xgb
xgb_model = xgb.XGBClassifier(n_jobs=multiprocessing.cpu_count()-2)

xgb_model.fit(X_train, y_train)

In [ ]:
print( 'train set performance:',roc_auc_score(y_train , xgb_model.predict_proba( X_train ) , multi_class='ovr' ) )
print( 'valid set performance:',roc_auc_score(y_valid , xgb_model.predict_proba( X_valid ) , multi_class='ovr') )

# "basic" usage 

In [ ]:
import optuna

## function to optimize

We need to write a function which :
 * takes a dictionary of parameters as arguments
 * returns a dictionnary containing the computed loss and a status value
 
Importantly, the loss is a value which we want to **minimize**, so in our case we will
use -AUC(validation data)

In [ ]:

def objective( trial ):
    """ function that takes an Optuna trial object 
    and returns a score 
    """
    
    ## suggesting a value for max_depth using previous information
    max_depth = trial.suggest_int( "max_depth" , 1, 16  )
    

    ## instanciate and train model
    xgb_model = xgb.XGBClassifier(n_jobs=multiprocessing.cpu_count()-2 , 
                                  max_depth = max_depth
                                 )
    xgb_model.fit(X_train, y_train)
    
    ## evaluate model
    roc_auc = roc_auc_score(y_valid , xgb_model.predict_proba( X_valid ) , multi_class='ovr' )
    
    return roc_auc


In [ ]:
%%time
study = optuna.create_study(direction="maximize")  # Create a new study.
study.optimize(objective, n_trials=5)  # Invoke optimization of the objective function.

## parameter search-space

From their [documentation](https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/002_configurations.html): 

For hyperparameter sampling, Optuna provides the following features:

 * `optuna.trial.Trial.suggest_categorical()` for categorical parameters
 * `optuna.trial.Trial.suggest_int()` for integer parameters
 * `optuna.trial.Trial.suggest_float()` for floating point parameters

With optional arguments of:
 * `step` :  discretize the search space (eg. sample in steps of 10 by 10)
 * `log` : sample in the logarithm space

In [ ]:
import optuna


def sampler_demo(trial):
    # Categorical parameter
    A = trial.suggest_categorical("A", ["choice1", "choice2"])

    # Integer parameter
    B = trial.suggest_int("B", 1, 3) # suggests 1, 2, or 3

    # Integer parameter (log)
    C = trial.suggest_int("C", 32, 512, log=True)

    # Integer parameter (discretized)
    D = trial.suggest_int("D", 10, 100, step=5) # suggests 10, 15, 20, 25, ...

    # Floating point parameter
    E = trial.suggest_float("E", 0.0, 1.0)

    # Floating point parameter (log)
    F = trial.suggest_float("F", 1e-5, 1e-2, log=True)

    # Floating point parameter (discretized)
    G = trial.suggest_float("G", 0.0, 1.0, step=0.1)




**exercise** : Complete the following function to suggest values for `n_estimators` and `eta`

* `n_estimators` : integer values between 1 and 1000 
* `eta` : float values sampled between $10^{-3}$ and $10^{2}$ in the log space

In [ ]:

def objective( trial ):
    """ function that takes an Optuna trial object 
    and returns a score 
    """
    
    ## suggesting a value for max_depth using previous information
    max_depth = trial.suggest_int( "max_depth" , 1, 16  )
    
    n_estimators = ... ## complete this 
    eta = ...          ## complete this 
    

    ## instanciate and train model
    xgb_model = xgb.XGBClassifier(n_jobs=multiprocessing.cpu_count()-2 , 
                                  max_depth = max_depth,
                                  n_estimators = n_estimators,
                                  eta = eta,
                                 )
    xgb_model.fit(X_train, y_train)
    
    ## evaluate model
    roc_auc = roc_auc_score(y_valid , xgb_model.predict_proba( X_valid ) , multi_class='ovr' )
    
    return roc_auc


In [ ]:
%%time
study = optuna.create_study(direction="maximize")  # Create a new study.
study.optimize(objective, n_trials=50)  # Invoke optimization of the objective function.

In [ ]:
trial = study.best_trial

print("  Value: {}".format(trial.value))
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

In [ ]:
study.trials[:5]

In [ ]:
study.trials_dataframe()

In [ ]:
from plotly.io import show
fig = optuna.visualization.plot_optimization_history(study)
show(fig)

In [ ]:
fig = optuna.visualization.plot_parallel_coordinate(study)
show(fig)

In [ ]:
fig = optuna.visualization.plot_slice(study, params=["max_depth", "n_estimators", 'eta'])
show(fig)

In [ ]:
fig = optuna.visualization.plot_contour(study, params=["n_estimators", 'eta'])
show(fig)

We can also explore optuna's results with [optuna-dashboard](https://optuna.github.io/optuna-dashboard/).

For this, we'd need to have setup as storage for our study object from the start, using something like:

```
study = optuna.create_study(storage="sqlite:///optuna_study.db",
                            direction="maximize")  
```

Nevertheless, here we are going to create a new study with storage, and copy our existing trials into it:

In [ ]:

# 1. Create a persistent target study
persistent_storage = "sqlite:///optuna_study.db"
target_study = optuna.create_study(
    study_name="my_saved_study",
    storage=persistent_storage,
    direction=study.direction,  # Match direction from your original study
    load_if_exists=True,
)

# 2. Add all trials from the in-memory study to the persistent storage
target_study.add_trials(study.trials)


In [ ]:
from optuna_dashboard import run_server
run_server(persistent_storage)

You can also go to their [web assembly optuna-dashboard](https://optuna.github.io/optuna-dashboard/) and load your sqlite object.

## warm start

Just call the `.optimize()` method again:

In [ ]:
study.optimize(objective, n_trials=5)  # Invoke optimization of the objective function.

## nested search spaces

Sometimes you will explore multiple models or have hyper-parameters whose values depend on each others.

You can organize the hyper-parameter suggestions using `if...else` and loops in order to produce a complex sampling space:

In [ ]:
from sklearn.linear_model import LogisticRegression
def objective_nested( trial ):
    """ function that takes an Optuna trial object 
    and returns a score 
    """

    model_name = trial.suggest_categorical('model_name', ['logistic_regression', 'xgboost'])
    if model_name == 'logistic_regression':
        LR_C = trial.suggest_float('LR_C', 1e-10, 1e10, log=True)
        model = LogisticRegression(C=LR_C)
    else:
        XB_max_depth = trial.suggest_int( "XB_max_depth" , 1, 16  )
        model = xgb.XGBClassifier(n_jobs=multiprocessing.cpu_count()-2 , 
                                  max_depth = XB_max_depth
                                  )
    model.fit(X_train, y_train)
    
    ## evaluate model
    roc_auc = roc_auc_score(y_valid , model.predict_proba( X_valid ) , multi_class='ovr' )
    
    return roc_auc


In [ ]:

study_nested = optuna.create_study(
    study_name="LR_vs_XGBOOST",
    storage="sqlite:///optuna_study_2.db",
    direction="maximize",
    load_if_exists = True
)

%time study_nested.optimize( objective_nested , n_trials=50)

In [ ]:
run_server('sqlite:///optuna_study_2.db')

## switching to cross-validation

You can switch the cross-validation scheme by modifying the objective function:

In [ ]:
from sklearn.model_selection import cross_val_score

def objective_CV( trial ):
    """ function that takes an Optuna trial object 
    and returns a score 
    """
    
    max_depth = trial.suggest_int( "max_depth" , 1, 16  )
    
    xgb_model = xgb.XGBClassifier(n_jobs=multiprocessing.cpu_count()-2 , 
                                  max_depth = max_depth
                                 )
        
    ## using cross_val_score instead of a simple train/validation scheme:
    roc_auc = cross_val_score(xgb_model, X_valid, y_valid , 
                    scoring = "roc_auc_ovr",
                    cv = 3 )
     
    
    return roc_auc.mean()


In [ ]:
%%time
study_cv = optuna.create_study(direction="maximize")  # Create a new study.
%time study_cv.optimize(objective_CV, n_trials=5)

## Other cool things:

 * [multi-objectives optimization](https://optuna.readthedocs.io/en/stable/tutorial/20_recipes/002_multi_objective.html)
 * using a [pruner to stop unpromising trials early](https://github.com/optuna/optuna-examples/blob/main/xgboost/xgboost_integration.py)